In [1]:
import pandas as pd
import json
from pathlib import Path
from load_gamelogs import _load_gamelogs, _load_player_types
from add_re_column import _add_re_column
from add_war_columns import _add_war_columns
from get_player_names import get_player_names
from get_single_game_stats import (
    get_hitting_game_stats, get_pitching_game_stats,
    get_hitting_team_game_stats, get_pitching_team_game_stats,
    get_hitting_combined_game_stats, get_pitching_combined_game_stats,
    get_single_game_records, get_team_game_records, get_combined_game_records,
    BATTER_STATS, PITCHER_STATS,
)

In [2]:
player_names = get_player_names()

# --- MLR Regular Season ---
league = 'mlr'
season = 12

df = _load_gamelogs(season, league)
df = _add_re_column(df, league)
df = _add_war_columns(df)

In [3]:
mlr_hitting_game = get_hitting_game_stats(df)
print(f'Rows: {len(mlr_hitting_game):,}  (player-game pairs)')
mlr_hitting_game.sort_values('HR', ascending=False).head(10)

Rows: 49,604  (player-game pairs)


,ID,Game ID,Season,Display Season,Session,Team,Franchise,Opponent,HR,3B,...,CS,H,TB,PA,R,RBI,RE24,WAR,WPA,Location
2303,56,2079,2,S2,8,TOR,TOR,MIN (S1-2),3,0,...,0,4,14,4,3,4,5.133270,0.528818,0.23,Away
41473,2722,10013,10,S10,1,OAK,OAK,SEA,3,0,...,0,3,12,3,3,4,3.679212,0.379041,0.7324,Away
33929,2219,2085,2,S2,8,SEA,SEA (S2),HOU,3,0,...,0,3,12,3,3,4,3.632782,0.374896,0.1867,Away
30785,1801,9063,9,S9,5,CHC,CHC,STL,3,0,...,0,3,12,4,3,5,4.414633,0.455713,0.3589,Away
40003,2646,10145,10,S10,10,PHI,PHI,NYM,3,0,...,0,3,12,3,3,4,3.731801,0.384300,0.2924,Home
37302,2487,8143,8,S8,10,OAK,OAK,ANA,3,0,...,0,4,14,4,4,7,6.137033,0.629032,0.2542,Away
37831,2517,7214,7,S7,15,OAK,OAK,DET,3,0,...,0,3,12,4,3,6,5.173312,0.533250,0.3235,Away
25524,686,7211,7,S7,15,MIL,MIL,CIN,3,0,...,0,3,12,3,3,3,3.000000,0.311939,0.3061,Away
24473,648,4141,4,S4,10,CLE,ANA,TEX,3,0,...,0,3,12,3,3,5,4.300398,0.441144,0.6701,Away
21801,530,12217,12,S12,15,MIN,MIN,SEA,3,0,...,0,3,12,3,3,4,3.615357,0.373365,0.1314,Home


In [4]:
mlr_pitching_game = get_pitching_game_stats(df)
print(f'Rows: {len(mlr_pitching_game):,}  (pitcher-game pairs)')
mlr_pitching_game.sort_values('SO', ascending=False).head(10)

Rows: 9,879  (pitcher-game pairs)


,ID,Game ID,Season,Display Season,Session,Team,Franchise,Opponent,HR,3B,...,H,BF,IP,ER,RE24,WAR,WPA,300+ Pitches,400+ Pitches,Location
5349,714,9054,9,S9,4,TOR,TOR,TBR,1,0,...,2,38,11.000000,1,-6.129891,0.776672,1.3062,17,6,Away
5075,646,4127,4,S4,9,TEX,CLE,SEA,2,0,...,5,27,6.000000,4,0.662347,0.053086,0.1542,12,0,Home
1362,127,4199,4,S4,14,KCR,KCR,ANA,3,0,...,6,27,6.000000,3,-0.337653,0.170763,0.2518,11,2,Away
1723,164,3070,3,S3,6,LAD,LAD,NYM,2,0,...,2,27,7.000000,3,-0.278146,0.147541,0.2513,12,7,Home
8193,2659,8192,8,S8,13,NYM,NYM,CIN,1,0,...,5,31,7.666667,2,-2.058242,0.328146,0.2898,11,3,Home
2976,275,6187,6,S6,13,CWS,CWS,CLE,1,0,...,3,24,6.000000,2,-1.427119,0.243097,0.0986,11,4,Away
8584,2744,11177,11,S11,12,BAL,BAL,NYY,1,0,...,4,22,6.000000,1,-2.596474,0.351833,0.2237,11,2,Away
9788,3192,11091,11,S11,7,ATL,ATL,ARI,4,0,...,5,30,6.333333,4,0.164272,0.109281,-0.0983,8,3,Away
5720,777,4222,4,S4,15,ARI,ARI,CIN,0,1,...,4,29,6.000000,4,0.662347,0.061924,-0.184,8,4,Home
7760,2523,7057,7,S7,4,MTL,MTL,PIT,2,0,...,4,31,8.000000,2,-4.936098,0.623681,0.9161,14,6,Home


In [5]:
id_to_name = player_names.set_index('ID')['Name'].to_dict()

def _fmt_holder(h):
    '''Format a player or team holder for display.'''
    game = f"{h['season']}.{h['session']}"
    prefix = 'vs' if h.get('location') == 'Home' else '@'
    if 'id' in h:
        name = id_to_name.get(h['id'], f"#{h['id']}")
        return f"{name} ({game}, {h['team']} {prefix} {h['opponent']})"
    else:
        return f"{h['team']} ({game} {prefix} {h['opponent']})"

def _fmt_combined(h):
    '''Format a combined (both-teams) holder for display.'''
    return f"{h['away']} @ {h['home']} ({h['season']}.{h['session']})"

def display_records(records, side, label):
    '''Print single-game records for player or team holders.'''
    print(f'=== {label} Single-Game Records: {side.title()} ===\n')
    for stat, rec in records[side].items():
        record_val = rec['record']
        if 'holders' in rec:
            holders_str = ' | '.join(_fmt_holder(h) for h in rec['holders'])
            print(f'{stat:15s}  {str(record_val):>8}   {holders_str}')
        else:
            mr = rec['most_recent']
            print(f'{stat:15s}  {str(record_val):>8}   '
                  f'{rec["tie_count"]}-way tie | most recent: {_fmt_holder(mr)}')

def display_records_combined(records, side, label):
    '''Print single-game records for combined (both-teams) holders.'''
    print(f'=== {label} Single-Game Records: {side.title()} ===\n')
    for stat, rec in records[side].items():
        record_val = rec['record']
        if 'holders' in rec:
            holders_str = ' | '.join(_fmt_combined(h) for h in rec['holders'])
            print(f'{stat:15s}  {str(record_val):>8}   {holders_str}')
        else:
            mr = rec['most_recent']
            print(f'{stat:15s}  {str(record_val):>8}   '
                  f'{rec["tie_count"]}-way tie | most recent: {_fmt_combined(mr)}')

mlr_records = get_single_game_records(mlr_hitting_game, mlr_pitching_game)
display_records(mlr_records, 'batter', 'MLR')

=== MLR Single-Game Records: Batter ===

2B                      3   21-way tie | most recent: Kuuma Kivi (S11.10, SFG @ ATL)
3B                      2   54-way tie | most recent: Connor Morgan (S12.13, KCR vs HOU)
Auto K                  2   30-way tie | most recent: Emer Bock (S8.10, PIT @ MIL)
BB                      3   56-way tie | most recent: Henry Thibodeau (S12.15, KCR @ CLE)
CS                      2   22-way tie | most recent: Primero Ultimo (S12.9, TBR @ SEA)
FO                      4   6-way tie | most recent: Dakota Carolina Montana (S10.15, PHI @ FLA)
GIDP                    3   Biff Sexy Hotbod (S10.15, MTL vs SFG)
H                       4   43-way tie | most recent: Tleyber Alfredo Cruz Fernandez III (S12.6, ATL @ SDP)
HR                      3   12-way tie | most recent: Ping Pong (S12.15, MIN vs SEA)
IBB                     3   Holden Summers (S3.8, ARI @ HOU)
LGO                     4   Chocolate Smegma (S1.2, MIL @ ATL)
PA                      6   19-way tie | mos

In [6]:
display_records(mlr_records, 'pitcher', 'MLR')

=== MLR Single-Game Records: Pitcher ===

2B                     10   Merodach Baladan (S9.10, DET vs CHC)
300+ Pitches           23   Chris Orosz (S10.15, CHC @ COL)
3B                      6   Jack Yakker (S5.15, DET vs TOR)
400+ Pitches           12   Tristan (S9.12, CIN @ STL) | Grand Slamdalf (S7.1, MIN @ CLE)
Auto BB                 3   Cal Tiberius Jr. (S3.9, CIN vs OAK) | Cal Tiberius Jr. (S3.3, ATL @ COL)
BB                      9   Skipper Studebaker (S11.12, PIT vs STL) | Willoughby Hoose (S5.10, HOU vs DET) | Snoop E. Dogg (S4.9, BOS vs NYY)
BF                     46   Tark Tarkington Jr (S9.4, TBR vs TOR)
CS                      3   17-way tie | most recent: Francis Swagger (S12.13, DET vs SEA)
DP                      5   Chris Orosz (S10.15, CHC @ COL) | Hank Murphy (S7.13, COL @ ARI)
ER                     14   Gary Appalachia (S12.5, CIN vs CHC)
FO                     12   Artanis Jones (S10.15, FLA vs PHI) | Tanzig Nordic (S10.11, NYM vs PIT)
H                      15 

In [7]:
mlr_hitting_team_game = get_hitting_team_game_stats(df)
mlr_pitching_team_game = get_pitching_team_game_stats(df)
print(f'Team hitting rows: {len(mlr_hitting_team_game):,}  (team-game pairs)')
print(f'Team pitching rows: {len(mlr_pitching_team_game):,}')
mlr_hitting_team_game.sort_values('HR', ascending=False).head(10)

Team hitting rows: 5,170  (team-game pairs)
Team pitching rows: 5,170


,Team,Opponent,Game ID,Season,Display Season,Session,HR,3B,2B,1B,...,CS,H,TB,PA,R,RBI,RE24,WAR,WPA,Location
983,CHC,STL,9063,9,S9,5,7,0,3,3,...,0,13,37,36,11,11,7.110969,0.839348,0.6917,Away
1846,DET,TEX,7089,7,S7,6,7,1,1,2,...,0,11,35,29,10,10,6.304555,0.749848,0.4266,Away
1241,CLE,HOU,4068,4,S4,5,7,0,4,3,...,0,14,39,34,13,13,9.662347,1.092087,0.4003,Away
846,CHC,ATL,7190,7,S7,13,7,1,1,6,...,0,15,39,35,17,17,13.920462,1.531337,0.5426,Home
2279,KCR,DET,7126,7,S7,9,7,0,1,3,...,0,11,33,30,9,9,5.304555,0.653828,0.435,Away
981,CHC,STL,8237,8,S8,16,6,0,0,3,...,0,9,27,30,14,14,10.754879,1.190453,0.4968,Home
4694,TBR,KCR,3084,3,S3,7,6,1,0,1,...,0,8,28,24,9,9,6.370130,0.734299,0.8501,Away
796,BOS,TBR,7079,7,S7,6,6,0,4,2,...,0,12,34,28,9,9,5.920462,0.703479,0.472,Home
4939,TEX,SEA,9150,9,S9,10,6,0,1,7,...,0,14,33,32,13,13,9.110969,1.028661,0.8591,Away
1134,CIN,SDP,11056,11,S11,4,6,0,2,7,...,1,15,35,36,10,10,6.403526,0.778002,0.534,Away


In [8]:
mlr_team_records = get_team_game_records(mlr_hitting_team_game, mlr_pitching_team_game)
display_records(mlr_team_records, 'batter', 'MLR Team')

=== MLR Team Single-Game Records: Batter ===

2B                     10   CHC (S9.10 @ DET)
3B                      6   TOR (S5.15 @ DET)
Auto K                  5   MIL (S2.18 vs STL)
BB                     12   MIN (S4.3 @ HOU)
CS                      4   BAL (S10.3 @ OAK) | BAL (S9.16 vs BOS) | BAL (S9.7 @ BOS) | SDP (S6.10 vs LAD)
FO                     12   PHI (S10.15 @ FLA) | PIT (S10.11 @ NYM) | LAD (S3.6 vs NYM)
GIDP                    5   STL (S12.3 @ CHC) | COL (S10.15 vs CHC) | ARI (S7.13 vs COL) | BOS (S7.10 vs TEX) | CWS (S7.5 @ DET)
H                      22   MIN (S4.3 @ HOU)
HR                      7   CHC (S9.5 @ STL) | CHC (S7.13 vs ATL) | KCR (S7.9 @ DET) | DET (S7.6 @ TEX) | CLE (S4.5 @ HOU)
IBB                     4   OAK (S7.16 vs SEA) | PIT (S5.1 vs SDP)
LGO                    12   ATL (S1.4 @ HOU) | MIL (S1.2 @ ATL)
PA                     50   CHC (S9.2 vs STL) | STL (S9.2 @ CHC) | MIN (S4.3 @ HOU)
PO                     12   MIN (S11.9 vs DET) | ARI (S5.8 @ MI

In [9]:
display_records(mlr_team_records, 'pitcher', 'MLR Team')

=== MLR Team Single-Game Records: Pitcher ===

2B                     10   DET (S9.10 vs CHC)
300+ Pitches           23   CHC (S10.15 @ COL)
3B                      6   DET (S5.15 vs TOR)
400+ Pitches           13   STL (S12.5 vs MIL) | HOU (S9.7 vs KCR) | DET (S9.3 @ OAK)
Auto BB                 5   SEA (S2) (S2.18 vs HOU)
BB                     12   HOU (S4.3 vs MIN)
BF                     50   CHC (S9.2 vs STL) | STL (S9.2 @ CHC) | HOU (S4.3 vs MIN)
CS                      4   OAK (S10.3 vs BAL) | BOS (S9.16 @ BAL) | BOS (S9.7 vs BAL) | LAD (S6.10 @ SDP)
DP                      5   CHC (S12.3 vs STL) | CHC (S10.15 @ COL) | COL (S7.13 @ ARI) | TEX (S7.10 @ BOS) | DET (S7.5 vs CWS)
ER                     22   HOU (S4.3 vs MIN)
FO                     12   FLA (S10.15 vs PHI) | NYM (S10.11 vs PIT) | NYM (S3.6 @ LAD)
H                      22   HOU (S4.3 vs MIN)
HR                      7   STL (S9.5 vs CHC) | ATL (S7.13 @ CHC) | DET (S7.9 vs KCR) | TEX (S7.6 vs DET) | HOU (S4.5 vs CLE)
I

In [10]:
mlr_hitting_combined_game = get_hitting_combined_game_stats(df)
mlr_pitching_combined_game = get_pitching_combined_game_stats(df)
print(f'Combined hitting rows: {len(mlr_hitting_combined_game):,}  (game pairs)')
print(f'Combined pitching rows: {len(mlr_pitching_combined_game):,}')
mlr_hitting_combined_game.sort_values('HR', ascending=False).head(10)

Combined hitting rows: 2,585  (game pairs)
Combined pitching rows: 2,585


,Game ID,Season,Display Season,Session,HR,3B,2B,1B,BB,IBB,...,CS,H,TB,PA,R,RBI,RE24,WAR,WPA,Teams
1799,9175,9,S9,12,10,3,4,5,5,0,...,1,22,62,62,19,19,11.221937,1.353759,1.0842,BOS @ CLE
1270,7126,7,S7,9,9,2,4,5,2,0,...,1,20,55,56,14,14,6.609109,0.891736,0.37,KCR @ DET
1355,7211,7,S7,15,9,1,1,5,1,0,...,0,16,46,53,13,13,5.609109,0.775817,1.0314,MIL @ CIN
316,3084,3,S3,7,9,3,0,1,3,1,...,0,13,46,46,14,14,8.740260,1.056924,1.2204,TBR @ KCR
2160,11056,11,S11,4,9,1,3,13,9,0,...,1,26,58,66,21,21,14.406464,1.686967,1.568,CIN @ SDP
1358,7214,7,S7,15,9,0,5,5,8,0,...,0,19,51,63,17,17,9.609109,1.211635,0.2512,OAK @ DET
1233,7089,7,S7,6,9,1,3,3,1,0,...,0,16,48,52,12,12,4.609109,0.671838,0.3532,DET @ TEX
1519,8135,8,S8,9,9,0,8,2,7,0,...,0,19,54,64,19,19,8.869979,1.132258,2.1806,TOR @ TEX
1890,10026,10,S10,2,9,2,6,6,6,0,...,1,23,60,64,19,19,12.334455,1.478086,0.8494,PIT @ SDP
492,4068,4,S4,5,9,0,5,3,6,0,...,0,17,49,57,15,15,8.324693,1.047159,0.2999,CLE @ HOU


In [11]:
mlr_combined_records = get_combined_game_records(mlr_hitting_combined_game, mlr_pitching_combined_game)
display_records_combined(mlr_combined_records, 'batter', 'MLR Combined')

=== MLR Combined Single-Game Records: Batter ===

2B                     14   ANA @ KCR (S4.1)
3B                      6   CIN @ CHC (S6.7) | TOR @ DET (S5.15)
Auto K                  5   ARI @ DET (S3.13) | STL @ MIL (S2.18)
BB                     15   SFG @ FLA (S5.10)
CS                      6   NYM @ BAL (S6.14)
FO                     21   PIT @ STL (S9.7)
GIDP                    8   TEX @ BOS (S7.10)
H                      30   MIN @ HOU (S4.3)
HR                     10   BOS @ CLE (S9.12)
IBB                     5   CWS @ DET (S7.5) | SDP @ PIT (S5.1)
LGO                    19   ATL @ HOU (S1.4)
PA                    100   STL @ CHC (S9.2)
PO                     18   SFG @ MIL (S11.7)
R                      24   SFG @ LAD (S6.3) | MIN @ HOU (S4.3)
RBI                    24   SFG @ LAD (S6.3) | MIN @ HOU (S4.3)
RE24             17.716949152542373   SFG @ LAD (S6.3)
RGO                    18   TOR @ TBR (S9.4)
SB                      9   SFG @ LAD (S6.3)
SO                     28  

In [12]:
display_records_combined(mlr_combined_records, 'pitcher', 'MLR Combined')

=== MLR Combined Single-Game Records: Pitcher ===

2B                     14   ANA @ KCR (S4.1)
300+ Pitches           44   CHC @ COL (S10.15)
3B                      6   CIN @ CHC (S6.7) | TOR @ DET (S5.15)
400+ Pitches           22   ANA @ HOU (S11.12) | STL @ CHC (S9.2)
Auto BB                 5   HOU @ SEA (S2) (S2.18)
BB                     15   SFG @ FLA (S5.10)
BF                    100   STL @ CHC (S9.2)
CS                      6   NYM @ BAL (S6.14)
DP                      8   TEX @ BOS (S7.10)
ER                     24   SFG @ LAD (S6.3) | MIN @ HOU (S4.3)
FO                     21   PIT @ STL (S9.7)
H                      30   MIN @ HOU (S4.3)
HR                     10   BOS @ CLE (S9.12)
IBB                     5   CWS @ DET (S7.5) | SDP @ PIT (S5.1)
IP                   25.1   STL @ CHC (S9.2)
LGO                    19   ATL @ HOU (S1.4)
PO                     18   SFG @ MIL (S11.7)
RE24             -11.44407912079222   STL @ CHC (S9.2)
RGO                    18   TOR @ TBR

In [13]:
# --- MLR Playoffs ---
league = 'mlr_playoff'
season = 11

df_po = _load_gamelogs(season, league)
df_po = _add_re_column(df_po, league)
df_po = _add_war_columns(df_po)

mlr_po_hitting_game = get_hitting_game_stats(df_po)
mlr_po_pitching_game = get_pitching_game_stats(df_po)
print(f'Playoff hitting rows: {len(mlr_po_hitting_game):,}')
print(f'Playoff pitching rows: {len(mlr_po_pitching_game):,}')

Playoff hitting rows: 2,020
Playoff pitching rows: 448


In [14]:
mlr_po_records = get_single_game_records(mlr_po_hitting_game, mlr_po_pitching_game, is_playoff=True)
display_records(mlr_po_records, 'batter', 'MLR Playoffs')

=== MLR Playoffs Single-Game Records: Batter ===

2B                      3   Queue Jay (S5.CS, OAK @ KCR)
3B                      2   Clarinet Clarinet II (S6.WC, SDP @ LAD) | Ray Dingerz (S4.DS, NYY vs BAL) | Dwayne Stevenson (S4.DS, NYM @ SFG)
Auto K                  1   7-way tie | most recent: Cujo (S6.PC, STL vs CWS)
BB                      2   45-way tie | most recent: Dakota Carolina Montana (S11.PC, PHI vs DET)
CS                      2   Vinny Vega (S10.CS, BAL @ TBR) | Jed I. Knight (S10.DS, ATL @ SFG) | Lil Lilith Lillian (S9.DS, BAL vs CLE) | Jack Parkman (S9.WC, TEX vs KCR) | Cam'ron Simmons (S5.PC, SDP @ OAK)
FO                      3   8-way tie | most recent: Randall Peppers (S9.CS, MTL @ COL)
GIDP                    3   Scotty Smalls (S4.PC, LAD @ KCR)
H                       4   6-way tie | most recent: Johnmaine Chupri (S9.DS, COL vs CHC)
HR                      2   10-way tie | most recent: Alvin and the Chipmunks Chipwrecked (S11.CS, DET @ TBR)
IBB                

In [15]:
display_records(mlr_po_records, 'pitcher', 'MLR Playoffs')

=== MLR Playoffs Single-Game Records: Pitcher ===

2B                      6   Gray Goose McGillicuddy (S11.WC, KCR @ BOS)
300+ Pitches           17   Judge Reinhold (S5.DS, SDP @ PIT)
3B                      3   Pauiie Wag (S8.WC, HOU vs KCR)
400+ Pitches           10   im sleve (S10.WC, PHI vs FLA)
Auto BB                 1   Superbone Threefinger (S3.DS, CLE vs NYY)
BB                      7   Otto von Pitchmark (S10.DS, MIL vs PHI) | Jefferson Steelflex (S5.WC, OAK vs TEX)
BF                     30   Skye Comet (S6.DS, STL @ COL)
CS                      5   Copernicus Hel (S10.CS, TBR vs BAL)
DP                      4   Mike Hawk (S9.WC, CLE @ BOS)
ER                      8   Mark Schihne (S10.DS, OAK @ TBR)
FO                      9   Judge Reinhold (S5.CS, SDP @ MTL) | Matt Himynamis (S5.DS, PIT vs SDP)
H                      11   Judge Reinhold (S6.CS, SDP @ STL)
HR                      5   Superbone Threefinger (S3.DS, CLE vs NYY)
IBB                     1   26-way tie | most r

In [16]:
mlr_po_hitting_team_game = get_hitting_team_game_stats(df_po)
mlr_po_pitching_team_game = get_pitching_team_game_stats(df_po)
mlr_po_team_records = get_team_game_records(mlr_po_hitting_team_game, mlr_po_pitching_team_game, is_playoff=True)
display_records(mlr_po_team_records, 'batter', 'MLR Playoffs Team')

=== MLR Playoffs Team Single-Game Records: Batter ===

2B                      7   BOS (S11.WC vs KCR)
3B                      3   KCR (S8.WC @ HOU) | MIN (S7.DS vs KCR) | SDP (S6.WC @ LAD) | NYM (S4.DS @ SFG)
Auto K                  2   CLE (S2.DS vs TOR)
BB                      7   PHI (S10.DS @ MIL) | OAK (S10.DS @ TBR) | PHI (S9.WC @ CHC) | TEX (S5.WC @ OAK)
CS                      5   BAL (S10.CS @ TBR)
FO                      9   6-way tie | most recent: CWS (S10.DS vs BAL)
GIDP                    4   BOS (S9.WC vs CLE) | LAD (S4.PC @ KCR)
H                      15   PHI (S9.WC @ CHC) | OAK (S8.DS vs SEA) | LAD (S4.PC @ KCR)
HR                      7   DET (S11.CS @ TBR)
IBB                     2   CWS (S10.DS vs BAL)
LGO                     8   STL (S11.CS @ PHI) | KCR (S7.DS @ MIN) | MIN (S5.WC @ NYY) | TEX (S3.CS @ CLE)
PA                     42   PHI (S9.WC @ CHC)
PO                      9   SFG (S4.DS vs NYM)
R                      13   NYY (S5.WC vs MIN)
RBI                

In [17]:
display_records(mlr_po_team_records, 'pitcher', 'MLR Playoffs Team')

=== MLR Playoffs Team Single-Game Records: Pitcher ===

2B                      7   KCR (S11.WC @ BOS)
300+ Pitches           17   CWS (S10.DS vs BAL) | SDP (S5.DS @ PIT)
3B                      3   HOU (S8.WC vs KCR) | KCR (S7.DS @ MIN) | LAD (S6.WC vs SDP) | SFG (S4.DS vs NYM)
400+ Pitches           11   ANA (S4.WC vs BAL)
Auto BB                 1   CLE (S3.DS vs NYY)
BB                      7   MIL (S10.DS vs PHI) | TBR (S10.DS vs OAK) | CHC (S9.WC vs PHI) | OAK (S5.WC vs TEX)
BF                     42   CHC (S9.WC vs PHI)
CS                      5   TBR (S10.CS vs BAL)
DP                      4   CLE (S9.WC @ BOS) | KCR (S4.PC vs LAD)
ER                     13   MIN (S5.WC @ NYY)
FO                      9   6-way tie | most recent: BAL (S10.DS @ CWS)
H                      15   CHC (S9.WC vs PHI) | SEA (S8.DS @ OAK) | KCR (S4.PC vs LAD)
HR                      7   TBR (S11.CS vs DET)
IBB                     2   BAL (S10.DS @ CWS)
IP                    9.0   BAL (S10.DS @ CWS) | CW

In [18]:
mlr_po_hitting_combined_game = get_hitting_combined_game_stats(df_po)
mlr_po_pitching_combined_game = get_pitching_combined_game_stats(df_po)
mlr_po_combined_records = get_combined_game_records(mlr_po_hitting_combined_game, mlr_po_pitching_combined_game, is_playoff=True)
display_records_combined(mlr_po_combined_records, 'batter', 'MLR Playoffs Combined')

=== MLR Playoffs Combined Single-Game Records: Batter ===

2B                      9   KCR @ BOS (S11.WC)
3B                      4   COL @ MIL (S11.WC) | PHI @ CHC (S9.WC) | KCR @ DET (S8.DS) | KCR @ MIN (S7.DS) | SDP @ LAD (S6.WC)
Auto K                  2   TOR @ CLE (S2.DS)
BB                     10   PHI @ CHC (S9.WC) | ATL @ STL (S6.WC) | LAD @ KCR (S4.PC)
CS                      6   BAL @ TBR (S10.CS)
FO                     17   BAL @ CWS (S10.DS)
GIDP                    5   ATL @ PHI (S10.CS) | PHI @ CHC (S9.WC) | STL @ COL (S6.DS)
H                      25   SEA @ OAK (S8.DS)
HR                     10   DET @ TBR (S11.CS)
IBB                     3   BAL @ CWS (S10.DS)
LGO                    13   TEX @ CLE (S3.CS)
PA                     73   PHI @ CHC (S9.WC)
PO                     12   CWS @ BAL (S6.DS)
R                      19   MIN @ NYY (S5.WC)
RBI                    19   MIN @ NYY (S5.WC)
RE24             11.80451127819549   MIN @ NYY (S5.WC)
RGO                    16   B

In [19]:
display_records_combined(mlr_po_combined_records, 'pitcher', 'MLR Playoffs Combined')

=== MLR Playoffs Combined Single-Game Records: Pitcher ===

2B                      9   KCR @ BOS (S11.WC)
300+ Pitches           31   PHI @ CHC (S9.WC)
3B                      4   COL @ MIL (S11.WC) | PHI @ CHC (S9.WC) | KCR @ DET (S8.DS) | KCR @ MIN (S7.DS) | SDP @ LAD (S6.WC)
400+ Pitches           18   FLA @ PHI (S10.WC)
Auto BB                 1   NYY @ CLE (S3.DS)
BB                     10   PHI @ CHC (S9.WC) | ATL @ STL (S6.WC) | LAD @ KCR (S4.PC)
BF                     73   PHI @ CHC (S9.WC)
CS                      6   BAL @ TBR (S10.CS)
DP                      5   ATL @ PHI (S10.CS) | PHI @ CHC (S9.WC) | STL @ COL (S6.DS)
ER                     19   MIN @ NYY (S5.WC)
FO                     17   BAL @ CWS (S10.DS)
H                      25   SEA @ OAK (S8.DS)
HR                     10   DET @ TBR (S11.CS)
IBB                     3   BAL @ CWS (S10.DS)
IP                   18.0   BAL @ CWS (S10.DS)
LGO                    13   TEX @ CLE (S3.CS)
PO                     12   CWS @ B

In [20]:
# Export JSON and cache files — run once records look correct
out_dir = Path('../docs/generated')
cache_dir = Path('../data')

# --- JSON output ---
# Merge all record types into one file per league
mlr_all_records = {
    'player': mlr_records,
    'team': mlr_team_records,
    'combined': mlr_combined_records,
}
with open(out_dir / 'mlr_single_game_records.json', 'w') as f:
    json.dump(mlr_all_records, f, indent=2)

mlr_po_all_records = {
    'player': mlr_po_records,
    'team': mlr_po_team_records,
    'combined': mlr_po_combined_records,
}
with open(out_dir / 'mlr_playoff_single_game_records.json', 'w') as f:
    json.dump(mlr_po_all_records, f, indent=2)

# --- Cache files (per-game stats for all seasons) ---
mlr_hitting_game.to_csv(cache_dir / 'mlr_hitting_game_stats_cache.csv', index=False)
mlr_pitching_game.to_csv(cache_dir / 'mlr_pitching_game_stats_cache.csv', index=False)
mlr_hitting_team_game.to_csv(cache_dir / 'mlr_hitting_team_game_stats_cache.csv', index=False)
mlr_pitching_team_game.to_csv(cache_dir / 'mlr_pitching_team_game_stats_cache.csv', index=False)
mlr_hitting_combined_game.to_csv(cache_dir / 'mlr_hitting_combined_game_stats_cache.csv', index=False)
mlr_pitching_combined_game.to_csv(cache_dir / 'mlr_pitching_combined_game_stats_cache.csv', index=False)

mlr_po_hitting_game.to_csv(cache_dir / 'mlr_playoff_hitting_game_stats_cache.csv', index=False)
mlr_po_pitching_game.to_csv(cache_dir / 'mlr_playoff_pitching_game_stats_cache.csv', index=False)
mlr_po_hitting_team_game.to_csv(cache_dir / 'mlr_playoff_hitting_team_game_stats_cache.csv', index=False)
mlr_po_pitching_team_game.to_csv(cache_dir / 'mlr_playoff_pitching_team_game_stats_cache.csv', index=False)
mlr_po_hitting_combined_game.to_csv(cache_dir / 'mlr_playoff_hitting_combined_game_stats_cache.csv', index=False)
mlr_po_pitching_combined_game.to_csv(cache_dir / 'mlr_playoff_pitching_combined_game_stats_cache.csv', index=False)

print('Done.')
print(f'  {out_dir}/mlr_single_game_records.json')
print(f'  {out_dir}/mlr_playoff_single_game_records.json')
print(f'  12 cache CSVs written to {cache_dir}/')

Done.
  ..\docs\generated/mlr_single_game_records.json
  ..\docs\generated/mlr_playoff_single_game_records.json
  12 cache CSVs written to ..\data/
